# StormEngine V8 — Stage 2 Processor-only training

This notebook keeps both Stage-1 Encoder/Decoder checkpoints frozen, trains a random-init ConvGRU L3K3 for each spatial candidate with seeds 42 and 43, and never reads the 2017 test split. Runs are sequential so one GPU is not oversubscribed. Interrupted formal runs resume from `last.pt`.

In [ ]:
from pathlib import Path
import subprocess
import sys

here = Path.cwd().resolve()
REPO = here if (here / 'pyproject.toml').is_file() else here.parent
assert (REPO / 'pyproject.toml').is_file(), REPO

PHASE = 'all'       # preflight | checks | train | all
SKIP_PILOT = False # set True only after a successful Stage-2 pilot was already run

command = [
    sys.executable, '-u', str(REPO / 'scripts' / 'run_v8_stage2.py'),
    '--phase', PHASE, '--device', 'cuda',
]
if SKIP_PILOT:
    command.append('--skip-pilot')
print('Running:', ' '.join(command), flush=True)
process = subprocess.Popen(
    command, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
assert process.stdout is not None
for line in process.stdout:
    print(line, end='', flush=True)
return_code = process.wait()
if return_code:
    raise subprocess.CalledProcessError(return_code, command)
print('Stage 2 workflow completed.', flush=True)